(Hassibi & Stork, NIPS 1992) ([__Optimal Brain Surgeon__: Second order derivatives for network
pruning: Optimal Brain Surgeon](https://proceedings.neurips.cc/paper/1992/file/303ed4c69846ab36c2904d3ba8573050-Paper.pdf)) — прямое продолжение OBD, исправляющее два главных недостатка предшественника: диагональное приближение гессиана и отсутствие компенсации оставшихся весов после удаления.

### Ключевая идея

Та же отправная точка — разложение лосса в ряд Тейлора около обученного минимума:

$$
\delta \mathcal{L} \approx \tfrac{1}{2} \delta\theta^\top H \, \delta\theta
$$

Линейный член снова зануляется в предположении сходимости, кубические отбрасываются. Но в отличие от OBD авторы *не* делают диагонального приближения — гессиан $H$ рассматривается как полная матрица.

Геометрически: многомерная функция вокруг точки $\theta$ приближается параболоидом, направления которого задаются собственными значениями матрицы H

Вторая принципиальная новация: давайте не откладывать дорогой fine-tuning на после прунинга, а будем сразу корректировать <u>все</u> веса модели при удалении каждого отедльного параметра. Они это называют "компенсацией ущерба"

<img src="img/obs.png" width=300>

Как это записать. 
- зануление веса $\theta_q$ эквивалентно движению в пространстве параметров, а конкретно проекции на одну из осей (q). Математически = наложение ограничения $\delta\theta_q + \theta_q = 0$. Или в матричной форме $e_q^\top \delta\theta + \theta_q = 0$, где $e_q$ — базисный вектор из нулей и единиц
- среди всех $\delta\theta$, удовлетворяющих этому ограничению, выбираем то, которое минимизирует $\delta \mathcal{L}$

Это задача квадратичной оптимизации с линейным ограничением, решается методом Лагранжа. Результат:

$$
\delta\theta = -\frac{\theta_q}{[H^{-1}]_{qq}} \, H^{-1} e_q \quad \text{- сюда двигаемся}
$$

$$
L_q = \tfrac{1}{2} \frac{\theta_q^2}{[H^{-1}]_{qq}} \quad \text{- новое значение оптимизируемой фкнкции потерь L}
$$

где $L_q$ — saliency веса $q$ (ожидаемое возрастание лосса при оптимальной компенсации), а $[H^{-1}]_{qq}$ — диагональный элемент *обратного* гессиана.

#### Как мы получили эти результаты Лагранжем

Мы хотим минимизировать квадратичный компонент $\tfrac{1}{2} \delta\theta^\top H \, \delta\theta$ функции потерь $L$ относительно возможных векторов сдвига $\delta \theta$ при наличии конкретного линейного ограничения: q-ая координата этого вектора точно должна быть сдвинута в 0 (так как мы ранее определили, что она бесполезна) $\delta\theta_q + \theta_q = 0$. То есть задаа выглядит так:

$$
\min_{\delta\theta} \; \tfrac{1}{2} \delta\theta^\top H \, \delta\theta \quad \text{при условии} \quad \delta\theta_q + \theta_q = 0
$$

Ограничение можно переписать как $e_q^\top \delta\theta = -\theta_q$, где $e_q$ — стандартный базисный вектор (нули везде, кроме позиции $q$, где единица)

Предполагаем, что $H$ симметрична и положительно определена ($H = H^\top$, все собственные значения $> 0$). Это гарантирует, что $H^{-1}$ существует и задача имеет единственное решение<br>
[уточнить почему прелполагаем]

<img src="img/lagrange.jpg" width=300>

Решается классическая задача квадратичной оптимизации с ограничением. Составляем лагранжиан с множителем $\lambda$:

$$
\mathcal{L}(\delta\theta, \lambda) = \tfrac{1}{2} \delta\theta^\top H \, \delta\theta + \lambda \, (e_q^\top \delta\theta + \theta_q)
$$

Необходимое условие оптимума - градиент лагранжиана по $\delta\theta$ должен быть нулевым:

$$
\frac{\partial \mathcal{L}}{\partial \delta\theta} = H \delta\theta + \lambda e_q = 0
$$

Отсюда:

$$
\delta\theta = -\lambda \, H^{-1} e_q
$$

Уже здесь можем определить *направление* движения — оно пропорционально $H^{-1} e_q$, а не самому $e_q$

Теперь подставим в ограничение, чтобы найти $\mu$:

$$
e_q^\top \delta\theta = -\lambda \, e_q^\top H^{-1} e_q = -\theta_q
$$

Заметим, что $e_q^\top H^{-1} e_q$ — это просто диагональный элемент $[H^{-1}]_{qq}$ (скалярное произведение базисного вектора с самим собой через матрицу выделяет $(q,q)$-элемент). Получаем:

$$
\lambda = \frac{\theta_q}{[H^{-1}]_{qq}}
$$

Подставляем обратно:

$$
\boxed{\delta\theta = -\frac{\theta_q}{[H^{-1}]_{qq}} \, H^{-1} e_q}
$$

И значение целевой функции в оптимуме:

$$
\tfrac{1}{2} \delta\theta^\top H \, \delta\theta = \tfrac{1}{2} \lambda^2 \, e_q^\top H^{-1} H H^{-1} e_q = \tfrac{1}{2} \lambda^2 \, [H^{-1}]_{qq} = \tfrac{1}{2} \frac{\theta_q^2}{[H^{-1}]_{qq}}
$$



### Итого

Во-первых, в saliency теперь стоит $\theta_q^2 / [H^{-1}]_{qq}$, а не $\theta_q^2 \cdot H_{qq}$. Это разные величины — для диагонального гессиана они совпадают (если $H$ диагональна, то $[H^{-1}]_{qq} = 1/H_{qq}$), но в общем случае нет. OBS использует структуру всего гессиана, не только диагонали.

Во-вторых, после удаления веса $q$ применяется поправка $\delta\theta = -\theta_q [H^{-1}]_{:,q} / [H^{-1}]_{qq}$ ко *всем* остальным весам. Это и есть «хирургия»: остальные параметры подкручиваются так, чтобы компенсировать потерю удалённого. В OBD после удаления веса требовалось дообучение с нуля; в OBS теоретически дообучение не нужно — компенсация уже встроена.

Отсюда и название: Optimal Brain Surgeon работает аккуратно, с микро-коррекцией соседей, в отличие от Optimal Brain Damage, который просто отрезает.

## Алгоритм

1. Обучить сеть до сходимости.
2. Вычислить $H^{-1}$.
3. Найти вес $q$ с минимальным saliency $L_q = \theta_q^2 / (2 [H^{-1}]_{qq})$.
4. Если $L_q$ меньше допустимого роста ошибки, удалить $\theta_q$ и применить поправку $\delta\theta = -\theta_q [H^{-1}]_{:,q} / [H^{-1}]_{qq}$ ко всем остальным весам.
5. Обновить $H^{-1}$ (есть рекуррентная формула, которая обновляет обратный гессиан без полного пересчёта).
6. Повторять с шага 3, пока удаление не приводит к недопустимому росту ошибки.

Дообучение между удалениями в исходной формулировке не предполагается — это отличает OBS от большинства современных методов прунинга, где fine-tuning обязателен.

---

#### Формула Шермана–Моррисона

Есть такая полезная алгебраическая формула, которая показывает, как можно "потоково" вычислить обратную матрицу $A^{-1}$, если к основной матрице $A$ добавляется некий update. Этот update должен быть ранга 1 (1-rank), то есть, например, outer product двух векторов $uu^T$. В этом случае обратную матрицу можно выразить через прибавку:

$$
(A + u u^\top)^{-1} = A^{-1} - \frac{A^{-1} u u^\top A^{-1}}{1 + u^\top A^{-1} u}
$$

---

### Практическая реализация

Теория теорией, но это нужно считать. Полученная выше формула требует вычисления обратного гессиана $H^{-1}$ на каждом шаге алгоритма. Полный гессиан $H$ для сети из $N$ весов — это матрица $N \times N$. Уже для скромной сети с $N = 10^4$ это $10^8$ элементов, для современной CNN с $N = 10^7$ — это $10^{14}$, что физически невозможно ни хранить, ни обращать

__Идея 1__<br>Авторы предложили при вычислении $H$ заменить его на outer-product приближение (свойство известное как Gauss-Newton приближение или Fisher-аппроксимация для классификационных задач). Оно говорит, что вторая производная $\approx$ произведение первых производных

$$
H \approx \frac{1}{P} \sum_{p=1}^{P} \nabla_\theta f^{(p)} \, (\nabla_\theta f^{(p)})^\top
$$

где $\nabla_\theta f^{(p)}$ — вектор градиентов функции потерь по всем параметрам на $p$-м обучающем примере. Это приближение хорошее в окрестности минимума и даёт положительно определённую матрицу.

__Идея 2__<br>К гессиану применяется формула Шермана-Моррисона для последовательного обновления $H^{-1}$ при добавлении каждого примера:

$$
H_{p+1}^{-1} = H_p^{-1} - \frac{H_p^{-1} \nabla f^{(p+1)} (\nabla f^{(p+1)})^\top H_p^{-1}}{P + (\nabla f^{(p+1)})^\top H_p^{-1} \nabla f^{(p+1)}}
$$

Это позволяет вычислять $H^{-1}$ за $O(P N^2)$ операций, что для маленьких сетей 1990-х было приемлемо. Для сетей 2010-х и позже — уже нет, и это главная причина, почему «классический» OBS долго оставался академическим методом.

## Эксперименты в оригинальной работе

Авторы тестировали OBS на тех же масштабах, что и OBD — небольшие MLP и CNN на простых задачах. Сравнения показали:

- OBS удаляет больше весов при той же потере точности, чем OBD и magnitude pruning.
- На некоторых задачах OBS работал без дообучения, тогда как OBD и magnitude требовали retraining для восстановления качества.
- На небольших задачах, где можно было честно посчитать полный гессиан, разница между диагональным OBD и полным OBS была заметной — диагональное приближение действительно теряло информацию.

Был один эффектный результат на задаче XOR / parity: OBS обнаружил, что в обученной сети есть веса с большим magnitude, удаление которых тем не менее не повышает лосс — потому что полный гессиан учитывал, как другие веса могут это скомпенсировать. Magnitude-based методы такое пропустили бы.

## Историческая роль и возрождение

Сразу после публикации OBS считался теоретически правильной версией прунинга, но на практике редко применимой из-за вычислительной стоимости. На протяжении 1990-х и 2000-х доминировал magnitude pruning как простой и работающий baseline, а методы второго порядка оставались нишевыми.

Возрождение случилось в 2020-х по двум причинам.
- Первая — выяснилось, что для очень больших моделей (LLM) дообучение после прунинга нереалистично дорогое, и one-shot подход с компенсацией стал крайне ценен. Это ровно то, что обещает OBS теоретически.
- Вторая — появились приближения обратного гессиана, масштабируемые на миллиарды параметров.

__WoodFisher__ (Singh & Alistarh, 2020) использует выход блочно-диагональное приближение Fisher-матрицы и формулу Вудбери для эффективного обращения. Каждый блок соответствует одному слою, что разумно структурно и вычислительно посильно. Это позволило применить OBS-подобный метод к ResNet-50 и BERT.

__M-FAC__ (Frantar, Kurtic & Alistarh, 2021) — ещё более эффективное приближение через скользящее окно градиентов и low-rank структуру. Дальнейшее снижение стоимости.

__SparseGPT__ (Frantar & Alistarh, 2023) — это, по сути, OBS, переформулированный для матриц весов больших трансформеров. Ключевая идея: вместо работы со всем гессианом сети, задача декомпозируется послойно. Для каждого линейного слоя отдельно решается reconstruction problem — найти разреженную матрицу весов, минимизирующую отклонение выхода слоя на калибровочной выборке. Эта подзадача имеет вид:

$$
\min_{\hat{W}} \| W X - \hat{W} X \|_F^2 \quad \text{s.t.} \quad \|\hat{W}\|_0 \leq k
$$

где $X$ — активации калибровочных примеров на входе слоя. Гессиан этой подзадачи — это $X X^\top$, матрица размера $d_{\text{in}} \times d_{\text{in}}$, что вполне обрабатываемо. Дальше применяется OBS-подобная процедура: последовательно удалять веса, обновляя оставшиеся столбцы $\hat{W}$ для компенсации.

Это позволило получить one-shot прунинг LLaMA-65B на 50% разреженности за несколько часов на одном GPU без всякого дообучения, с минимальной потерей perplexity.

## Резюме

OBS отличается от OBD двумя вещами: использует полный гессиан вместо диагонального приближения, и компенсирует удаление веса коррекцией оставшихся параметров вместо полного дообучения. Saliency формулируется через обратный гессиан: $L_q = \theta_q^2 / (2 [H^{-1}]_{qq})$. В оригинале метод был ограничен размерами сетей 1990-х из-за стоимости работы с гессианом, но идея оказалась глубоко правильной — современные state-of-the-art методы прунинга LLM (SparseGPT в первую очередь) — это, по существу, OBS, адаптированный к масштабу через послойную декомпозицию и эффективные приближения обратного гессиана

---

### Примеры использования аналогичного подхода в математике

Формула OBS — это решение классической задачи минимизации квадратичной формы при линейном ограничении методом Лагранжа. Структура «возмущение $\propto H^{-1} \cdot$ вектор ограничения» с saliency $\propto (\text{вектор ограничения})^2 / [H^{-1}]_{qq}$ возникает в десятке других мест: метод Ньютона, условные гауссовские распределения, ограниченная регрессия, Калмановская фильтрация, influence functions, Шуровы дополнения. Это одна из фундаментальных конструкций в линейной алгебре, которая всякий раз говорит: «когда задача квадратичная и ограничение линейное, оптимум находится через обратный гессиан, проектируемый на направление ограничения, и стоимость соблюдения ограничения измеряется через диагональ обратного гессиана».

#### Базовая формула шага метода Ньютона

$$
\delta\theta = -H^{-1} g
$$

где $g = \nabla f$ — градиент. Это решение задачи *безусловной* минимизации квадратичного приближения функции:

$$
\min_{\delta\theta} \; g^\top \delta\theta + \tfrac{1}{2} \delta\theta^\top H \delta\theta
$$

Здесь в отличие от OBS нет ограничения, мы просто ищем минимум локального параболоида. Но геометрически очень близко: Ньютон идёт «по самому крутому спуску в $H$-метрике», что даёт $-H^{-1} g$ вместо обычного $-g$ (который был бы крутым спуском в евклидовой метрике).

Это как раз обоснование, почему натуральный градиент (Amari, 1998) и метод Ньютона часто работают лучше обычного SGD на плохо обусловленных задачах: они адаптируются к локальной геометрии.

#### Условные распределения многомерного нормального

Пусть $X \sim \mathcal{N}(0, \Sigma)$ — многомерное нормальное распределение с ковариационной матрицей $\Sigma$. Тогда условное среднее $X_{-q}$ при условии $X_q = x_q$ имеет вид:

$$
\mathbb{E}[X_{-q} \mid X_q = x_q] = \frac{x_q}{[\Sigma^{-1}]_{qq}} \cdot (-[\Sigma^{-1}]_{:,q})_{-q}
$$

(формула в некотором сокращении, но структура та же). Это в точности структура OBS-формулы, потому что задача математически идентична: «при заданном значении одной координаты, как должны сдвинуться остальные, чтобы минимизировать $X^\top \Sigma^{-1} X$ — что является логарифмом плотности?». Гауссовский MAP-вывод и OBS-компенсация — это одна и та же задача.

Конкретно, обратная ковариационная матрица $\Sigma^{-1}$ называется *матрицей точности* (precision matrix), и её внедиагональные элементы характеризуют условные зависимости между переменными. Это тесно связано с теорией графических моделей.

### Линейная регрессия с ограничениями

Задача OLS с линейным ограничением $A\beta = b$:

$$
\min_\beta \; \|y - X\beta\|^2 \quad \text{при} \quad A\beta = b
$$

Решение использует ту же лагранжеву технику. Гессиан здесь $X^\top X$, и в формуле для условного оптимума $\beta$ появляется $(X^\top X)^{-1} A^\top$ — структурно полная аналогия с $H^{-1} e_q$ из OBS.

Это базовая конструкция в эконометрике (restricted least squares) и в задачах смешанных моделей.

### Калмановская фильтрация

Шаг обновления в фильтре Калмана: дано априорное состояние $\hat{x}^-$ с ковариацией $P^-$, поступает измерение $z = H x + v$ с шумом ковариации $R$. Апостериорная оценка:

$$
\hat{x}^+ = \hat{x}^- + K (z - H \hat{x}^-)
$$

где Калмановский gain $K = P^- H^\top (H P^- H^\top + R)^{-1}$.

Это снова та же структура: «обновление состояния пропорционально ковариации, применённой к остатку». Калман — это, по сути, рекурсивный условный оптимум в гауссовской модели, что напрямую соответствует условной задаче из OBS, только в баесовском сеттинге.

### Метод сопряжённых градиентов и проекции в $H$-метрике

Если делать прунинг сразу нескольких весов, OBS обобщается естественным образом. Допустим, удаляются веса с индексами $S = \{q_1, q_2, \ldots, q_k\}$. Ограничение становится матричным: $E_S^\top \delta\theta = -\theta_S$, где $E_S$ — матрица из соответствующих базисных столбцов.

Решение:

$$
\delta\theta = -H^{-1} E_S \, ([E_S^\top H^{-1} E_S]^{-1}) \, \theta_S
$$

Это в точности формула *проекции в $H$-метрике* на аффинное подпространство. Из-за этого алгоритмы итеративного прунинга нескольких весов имеют структуру, идентичную методу сопряжённых градиентов с активным набором ограничений (active-set methods в QP).